# Phase 5 — Model Calibration
**Home Credit Default Risk**

Calibration corrects the *meaning* of the model's output probability without changing its rank order (AUC stays the same).  
After this phase, a score of `0.15` will genuinely mean ~15% probability of default.

**Inputs required from your Phase 4 notebook:**
- `oof_preds` — raw OOF scores from the ensemble (shape: `n_train,`)
- `test_preds` — averaged raw scores on test set (shape: `n_test,`)
- `y` — true labels (shape: `n_train,`)
- `test` — original test DataFrame (for `SK_ID_CURR`)

## 0. Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import joblib
import warnings
warnings.filterwarnings('ignore')

from sklearn.calibration import calibration_curve
from sklearn.isotonic import IsotonicRegression
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, brier_score_loss

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({'figure.dpi': 130, 'font.size': 11})

print('All imports OK')

## 1. Load Phase 4 Outputs

Replace the paths below with wherever you saved your Phase 4 results.  
If you ran everything in one notebook, just make sure the variables are in scope.

In [ ]:
# ── Option A: load from saved files ──────────────────────────────────────────
# oof_preds  = np.load('phase4_oof_preds.npy')
# test_preds = np.load('phase4_test_preds.npy')
# y          = pd.read_csv('train_labels.csv')['TARGET'].values
# test       = pd.read_csv('application_test.csv')

# ── Option B: variables already in scope from Phase 4 ────────────────────────
# oof_preds, test_preds, y, test are assumed to exist

print(f'OOF predictions  : {oof_preds.shape}  | range [{oof_preds.min():.4f}, {oof_preds.max():.4f}]')
print(f'Test predictions : {test_preds.shape} | range [{test_preds.min():.4f}, {test_preds.max():.4f}]')
print(f'Labels           : {y.shape}  | default rate {y.mean():.4f}')

## 2. Diagnose — How Miscalibrated Is the Raw Model?

A **reliability diagram** (calibration curve) plots:
- X-axis: mean predicted probability per bin
- Y-axis: actual fraction of positives in that bin

A perfectly calibrated model follows the diagonal.  
GBDTs like LightGBM typically curve *above* the diagonal — they are overconfident.

In [ ]:
def reliability_diagram(y_true, y_prob, label, ax, n_bins=10, color='steelblue'):
    """Plot a reliability diagram (calibration curve) on a given axis."""
    fraction_pos, mean_pred = calibration_curve(y_true, y_prob, n_bins=n_bins)
    ax.plot(
        mean_pred, fraction_pos,
        marker='o', linewidth=2, markersize=5,
        label=label, color=color
    )
    ax.plot([0, 1], [0, 1], 'k--', linewidth=0.9, label='Perfect calibration', alpha=0.5)
    ax.set_xlabel('Mean predicted probability')
    ax.set_ylabel('Fraction of positives (actual)')
    ax.legend(fontsize=9)
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)

fig, ax = plt.subplots(figsize=(6, 5))
reliability_diagram(y, oof_preds, 'Raw ensemble (OOF)', ax, color='#E07B5A')
ax.set_title('Reliability Diagram — Before Calibration')
plt.tight_layout()
plt.savefig('calibration_before.png', bbox_inches='tight')
plt.show()

# Brier score: measures combined calibration + discrimination (lower = better)
brier_raw = brier_score_loss(y, oof_preds)
auc_raw   = roc_auc_score(y, oof_preds)
print(f'\nRaw model — AUC: {auc_raw:.5f} | Brier: {brier_raw:.5f}')
print(f'Baseline Brier (always predict mean): {brier_score_loss(y, np.full_like(y, y.mean(), dtype=float)):.5f}')

## 3. Fit Both Calibration Methods

| Method | How it works | Best when |
|---|---|---|
| **Isotonic regression** | Non-parametric monotone fit | Large datasets (>1k positives) |
| **Platt scaling** | Logistic regression on raw scores | Small datasets, more stable |

> ⚠️ Always calibrate on **OOF predictions** — they are already out-of-sample.  
> Fitting on training scores would overfit the calibrator.

In [ ]:
# ── Method A: Isotonic Regression ─────────────────────────────────────────────
iso_reg = IsotonicRegression(out_of_bounds='clip')
iso_reg.fit(oof_preds, y)

oof_cal_iso  = iso_reg.predict(oof_preds)
test_cal_iso = iso_reg.predict(test_preds)

# ── Method B: Platt Scaling (Logistic Regression) ─────────────────────────────
platt = LogisticRegression(C=1.0, solver='lbfgs')
platt.fit(oof_preds.reshape(-1, 1), y)

oof_cal_platt  = platt.predict_proba(oof_preds.reshape(-1, 1))[:, 1]
test_cal_platt = platt.predict_proba(test_preds.reshape(-1, 1))[:, 1]

print('Both calibrators fitted.')

## 4. Compare — Before vs After

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: reliability diagram
reliability_diagram(y, oof_preds,      'Raw',              axes[0], color='#E07B5A')
reliability_diagram(y, oof_cal_iso,    'Isotonic',         axes[0], color='#2EA87E')
reliability_diagram(y, oof_cal_platt,  'Platt scaling',    axes[0], color='#5B8DEF')
axes[0].set_title('Reliability Diagram — Comparison')

# Right: score distribution shift
axes[1].hist(oof_preds,     bins=60, alpha=0.5, label='Raw',        color='#E07B5A', density=True)
axes[1].hist(oof_cal_iso,   bins=60, alpha=0.5, label='Isotonic',   color='#2EA87E', density=True)
axes[1].hist(oof_cal_platt, bins=60, alpha=0.5, label='Platt',      color='#5B8DEF', density=True)
axes[1].axvline(y.mean(), color='black', linestyle='--', linewidth=1, label=f'Base rate ({y.mean():.3f})')
axes[1].set_xlabel('Predicted probability of default')
axes[1].set_ylabel('Density')
axes[1].set_title('Score Distribution Shift After Calibration')
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.savefig('calibration_comparison.png', bbox_inches='tight')
plt.show()

## 5. Metrics Table

- **AUC** should be identical (or near-identical) across all methods — calibration preserves rank order
- **Brier score** should drop after calibration — this is the signal that calibration worked
- **Mean prediction** should be close to the training base rate (~0.08)

In [ ]:
methods = {
    'Raw':              oof_preds,
    'Isotonic':         oof_cal_iso,
    'Platt scaling':    oof_cal_platt,
}

rows = []
for name, preds in methods.items():
    rows.append({
        'Method':       name,
        'AUC':          roc_auc_score(y, preds),
        'Brier Score':  brier_score_loss(y, preds),
        'Mean Pred':    preds.mean(),
        'Std Pred':     preds.std(),
    })

results_df = pd.DataFrame(rows).set_index('Method')
print(f'Training base rate: {y.mean():.5f}\n')
print(results_df.round(5).to_string())

print('\n→ AUC should be the same across all methods (rank order preserved)')
print('→ Brier Score should DECREASE after calibration')
print('→ Mean Pred should be close to base rate after calibration')

## 6. Choose and Save the Calibrator

**Decision rule:**
- Home Credit has ~307k samples and ~24k positives → use **Isotonic**
- If your positive class were <1k samples → use **Platt**

Pick whichever has the lower Brier score above.

In [ ]:
class CreditCalibrator:
    """
    Wraps raw ensemble scores → calibrated probabilities of default (PD).
    Fit once on OOF predictions, save, then reload for production inference.
    """
    def __init__(self, method='isotonic'):
        assert method in ('isotonic', 'platt'), "method must be 'isotonic' or 'platt'"
        self.method      = method
        self._calibrator = None
        self._base_rate  = None

    def fit(self, raw_scores: np.ndarray, y_true: np.ndarray):
        self._base_rate = y_true.mean()
        if self.method == 'isotonic':
            self._calibrator = IsotonicRegression(out_of_bounds='clip')
            self._calibrator.fit(raw_scores, y_true)
        else:
            self._calibrator = LogisticRegression(C=1.0, solver='lbfgs')
            self._calibrator.fit(raw_scores.reshape(-1, 1), y_true)
        return self

    def predict_proba(self, raw_scores: np.ndarray) -> np.ndarray:
        """Return calibrated probability of default for each applicant."""
        if self._calibrator is None:
            raise RuntimeError('Call .fit() before .predict_proba()')
        if self.method == 'isotonic':
            return self._calibrator.predict(raw_scores)
        else:
            return self._calibrator.predict_proba(raw_scores.reshape(-1, 1))[:, 1]

    def save(self, path='credit_calibrator.joblib'):
        joblib.dump({'method': self.method, 'calibrator': self._calibrator,
                     'base_rate': self._base_rate}, path)
        print(f'Calibrator saved → {path}')

    @classmethod
    def load(cls, path='credit_calibrator.joblib'):
        data = joblib.load(path)
        obj  = cls(method=data['method'])
        obj._calibrator = data['calibrator']
        obj._base_rate  = data['base_rate']
        print(f'Calibrator loaded ← {path}  (method={obj.method}, base_rate={obj._base_rate:.4f})')
        return obj


# Fit and save
calibrator = CreditCalibrator(method='isotonic')
calibrator.fit(oof_preds, y)
calibrator.save('credit_calibrator.joblib')

# Quick reload sanity check
cal_loaded = CreditCalibrator.load('credit_calibrator.joblib')
assert np.allclose(
    calibrator.predict_proba(oof_preds[:100]),
    cal_loaded.predict_proba(oof_preds[:100])
), 'Reload mismatch!'
print('Reload check passed ✓')

## 7. Final Calibrated Predictions

Apply the saved calibrator to test predictions and build the final submission.

In [ ]:
# Apply to test
test_preds_calibrated = calibrator.predict_proba(test_preds)

# Sanity checks
print('─── Calibrated test predictions ─────────────────')
print(pd.Series(test_preds_calibrated).describe().round(4))
print(f'\nTraining base rate        : {y.mean():.4f}')
print(f'Mean calibrated test pred : {test_preds_calibrated.mean():.4f}  ← should be close')
assert test_preds_calibrated.min() >= 0.0, 'Negative probabilities — check out_of_bounds'
assert test_preds_calibrated.max() <= 1.0, 'Probabilities > 1 — check out_of_bounds'
print('All sanity checks passed ✓')

In [ ]:
# Score distribution: raw vs calibrated side by side
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

for ax, preds, title, color in [
    (axes[0], test_preds,            'Raw ensemble scores',          '#E07B5A'),
    (axes[1], test_preds_calibrated, 'Calibrated PD (isotonic)',     '#2EA87E'),
]:
    ax.hist(preds, bins=80, color=color, alpha=0.8, edgecolor='none')
    ax.axvline(preds.mean(), color='black', linestyle='--', linewidth=1.2,
               label=f'Mean: {preds.mean():.4f}')
    ax.set_xlabel('Score')
    ax.set_ylabel('Count')
    ax.set_title(title)
    ax.legend(fontsize=9)

plt.suptitle('Test Set Score Distribution: Before vs After Calibration', y=1.01, fontsize=12)
plt.tight_layout()
plt.savefig('score_distribution.png', bbox_inches='tight')
plt.show()

In [ ]:
# Build submission
submission = pd.DataFrame({
    'SK_ID_CURR':   test['SK_ID_CURR'],
    'TARGET':       test_preds_calibrated,   # Calibrated PD — use this
    'TARGET_RAW':   test_preds,              # Raw score — keep for debugging
})
submission.to_csv('phase5_calibrated_submission.csv', index=False)

print(f'Submission saved: {submission.shape}')
print(submission.head(10).round(4))

## 8. Summary

| | Raw Ensemble | After Isotonic Calibration |
|---|---|---|
| **AUC** | Same | Same (rank order preserved) |
| **Brier Score** | Higher | Lower ✓ |
| **Mean prediction** | Off from base rate | ≈ base rate ✓ |
| **Interpretation** | Ranking score | True probability of default |

**Files produced:**
- `credit_calibrator.joblib` — the fitted calibrator (load this in production)
- `phase5_calibrated_submission.csv` — final submission with calibrated PD
- `calibration_before.png`, `calibration_comparison.png`, `score_distribution.png`

---
**Next:** Phase 6 — Drift monitoring, per-applicant SHAP explanations, and model card.